In [ ]:
from datetime import datetime
import os 
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:

    pass

PASSWORD_EDU = os.getenv("EDUPAGE_PASSWORD", "")

if PASSWORD_EDU is not None:
    PASSWORD_EDU = PASSWORD_EDU.strip()


if not PASSWORD_EDU:
    raise SystemExit("PASSWORD not found in environment. Add it to your .env (PASSWORD=...) or set it in the OS environment.")
from edupage_api import Edupage
from edupage_api.timeline import EventType

edupage = Edupage()
edupage.login(
    "KristianSiska",
    PASSWORD_EDU,
    "spsezoska",
)

notifications = edupage.get_notifications()

homework_not_due = 0
@dataclass
class Homework:
    event_id: int
    timestamp: datetime
    text: str
    author: Optional[str] = None
    recipient: Optional[str] = None
    event_type: Optional[Any] = None
    additional_data: Dict[str, Any] = field(default_factory=dict)

    # Common additional_data fields
    predmetid: Optional[str] = None
    triedaid: Optional[str] = None
    skupinyids: List[str] = field(default_factory=list)
    nazov: Optional[str] = None
    popis: Optional[str] = None
    date: Optional[str] = None
    due_date: Optional[datetime] = None
    typ: Optional[Any] = None
    id: Optional[str] = None
    skupiny: List[str] = field(default_factory=list)
    zmena: Optional[bool] = None
    oldVals: Optional[Dict[str, Any]] = None
    superid: Optional[str] = None
    etestCards: Optional[int] = None
    attachements: Optional[Any] = None
    planid: Optional[str] = None

    # Other timeline flags
    is_done: Optional[bool] = None
    done_at: Optional[datetime] = None
    is_starred: Optional[bool] = None
    reaction_count: Optional[int] = None
    created_at: Optional[datetime] = None
    is_removed: Optional[bool] = None

    @classmethod
    def from_timeline_event(cls, event) -> "Homework":
        ad = getattr(event, "additional_data", {}) or {}
        old_vals = ad.get("oldVals")

        # prefer date from additional_data, fall back to oldVals when present
        date_str = ad.get("date")
        if not date_str and isinstance(old_vals, dict):
            date_str = old_vals.get("date")

        due_date = None
        if date_str:
            try:
                due_date = datetime.strptime(date_str, "%Y-%m-%d")
            except Exception:
                due_date = None

        nazov = ad.get("nazov") or ad.get("title")

        return cls(
            event_id=getattr(event, "event_id", None),
            timestamp=getattr(event, "timestamp", None),
            text=getattr(event, "text", ""),
            author=getattr(event, "author", None),
            recipient=getattr(event, "recipient", None) or ad.get("recipient"),
            event_type=getattr(event, "event_type", None),
            additional_data=ad,
            predmetid=ad.get("predmetid"),
            triedaid=ad.get("triedaid"),
            skupinyids=ad.get("skupinyids") or [],
            nazov=nazov,
            popis=ad.get("popis"),
            date=date_str,
            due_date=due_date,
            typ=ad.get("typ"),
            id=ad.get("id"),
            skupiny=ad.get("skupiny") or [],
            zmena=ad.get("zmena"),
            oldVals=old_vals if isinstance(old_vals, dict) else None,
            superid=ad.get("superid"),
            etestCards=ad.get("etestCards"),
            attachements=ad.get("attachements") or ad.get("attachments"),
            planid=ad.get("planid"),
            is_done=getattr(event, "is_done", None),
            done_at=getattr(event, "done_at", None),
            is_starred=getattr(event, "is_starred", None),
            reaction_count=getattr(event, "reaction_count", None),
            created_at=getattr(event, "created_at", None),
            is_removed=getattr(event, "is_removed", None),
        )


homework = list(filter(lambda x: x.event_type == EventType.HOMEWORK, notifications))
print(homework, file=open("homework_debug.txt", "w", encoding="utf-8"))
for ev in homework:
    hw = Homework.from_timeline_event(ev)

    hw_title = hw.nazov
    hw_class = hw.recipient

    text = f"Homework from {hw.timestamp}\n"
    hw_text = hw_title or hw.text.replace("\n", " ")
    text += f"{hw_text}\n{hw_class}\n"

    if hw.due_date:
        now = datetime.now()
        text += f"\nDue to: {hw.due_date} -> "

        if hw.due_date > now:
            text += "DUE"
        else:
            homework_not_due += 1
            text += "UNFINISHED"

    print(text)
    print("—")

print(f"You have {homework_not_due} unfinished homework assignments.")



